# YZ50 — Week 4: Neural Network Language Model

- A **neural network language model** predicts the next character by using a fixed **context window** of previous characters instead of relying on a simple bigram table.

- Each character is represented by a learnable **embedding vector**, and the embeddings of the previous three characters are combined and passed through an **MLP** to predict the next character.

- The model is trained using **minibatches**, **gradient descent**, and a **train / dev / test split** to measure how well it generalizes to unseen data.

- **Tanh saturation** can cause gradients to become very small when activations approach -1 or 1. Proper **weight initialization**, such as **Kaiming initialization**, helps keep activations and gradients in a useful range.

- **BatchNorm** normalizes hidden-layer activations during training, helping stabilize the network and improve the training process.

In this week, the simple counting-based language model is replaced with a **neural network language model**, while also exploring how **embeddings, initialization, activations, and BatchNorm** affect the learning process.

# Continuing to Build Makemore: Character-Level Language Model with MLP

---

For this setup, the learnable parameters are the **weights (W)**, **biases (b)**, and the **embedding lookup table (C)**. All of them are updated through backpropagation and gradient descent to minimize the loss.


# 1. Part 2 ile başla. Önceki üç harfi bağlam alan veri setini kur (X: 3 harf indeksi, Y: sıradaki harf). Embedding tablosunu (27x2) oluştur indeksleme ile embedding'leri çek.

In [115]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt


In [116]:
# read in all the words
words = open('../data/names.txt', 'r').read().splitlines() # # Read the names from the file and split them into a list
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [117]:
len(words)

32033

## 1.1 Create the lookup tables

In [118]:
# build the vocabulary of characters and mappings to/from integers
# Create a lookup table to map characters to indices (0-25 for 'a'-'z', 26 for start/end tokens)
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


## 1.2 Construct the dataset

In [119]:
# build the dataset

block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], [] # X is the input data (context) for the neural network, Y is the target data (next character) for each input context
for w in words[:5]:

  print(w)
  context = [0] * block_size # initialize the context with zeros (start tokens)
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    print(''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # update the context by removing the first character and adding the new character, this works like a sliding window
    #print('context:', context)

X = torch.tensor(X)
Y = torch.tensor(Y)

# For each word, it creates a context of length `block_size` (initialized with zeros) and iterates through each character in the word (plus a period to indicate the end of the word). For each character, it appends the current context to `X` and the index of the character to `Y`. After processing each character, it updates the context by removing the first character and adding the new character, effectively creating a sliding window of context for predicting the next character.

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [120]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

## 1.3 Create the embeddings, embedding lookup table C

In [121]:
# Create the embedding table `C` with random values.
C = torch.randn((27,2)) # 27 rows for each character in the vocabulary (26 letters + 1 for the period), and 2 columns for the embedding dimension. Each row corresponds to a character's embedding vector.

In [122]:
C

tensor([[ 1.0696, -0.5928],
        [ 0.6969,  0.8277],
        [ 1.1964, -1.2858],
        [ 0.6886, -0.9722],
        [-1.0668, -0.2882],
        [ 0.1072, -0.7512],
        [ 0.8660, -0.6971],
        [-0.8521, -0.1696],
        [ 0.4154,  1.3483],
        [-0.0149, -0.7220],
        [-1.0962, -0.4165],
        [ 1.9601,  0.2844],
        [ 1.7781,  0.1825],
        [-1.0783, -0.6704],
        [ 1.2558, -0.2594],
        [-0.0637, -0.8183],
        [-1.0665,  2.0142],
        [ 0.5572,  0.3665],
        [ 0.8207,  0.7273],
        [-1.1740, -0.1605],
        [-1.0991, -1.7879],
        [-1.4667,  1.4619],
        [-0.7590,  0.2823],
        [ 1.2590, -1.4486],
        [ 0.3992,  0.4022],
        [-0.0640,  0.5078],
        [-0.5599, -0.6466]])

In [123]:
C[5] # much faster than doing one-hot encoding and matrix multiplication, but we will do that next to show that it is equivalent

tensor([ 0.1072, -0.7512])

In [124]:
F.one_hot(torch.tensor(5), num_classes=27).float() @ C # this is the same as C[5]

tensor([ 0.1072, -0.7512])

In [125]:
C[torch.tensor([5, 0, 2])] # this gives us the embeddings for the characters with indices 5, 0, and 2

tensor([[ 0.1072, -0.7512],
        [ 1.0696, -0.5928],
        [ 1.1964, -1.2858]])

In [126]:
X[2] # this gives us the indices of the characters in the context of the 3rd training example (index 2)

tensor([ 0,  5, 13])

In [127]:
C[X[2]] # this gives us the embeddings for the characters in the context of the 3rd training example (index 2)

tensor([[ 1.0696, -0.5928],
        [ 0.1072, -0.7512],
        [-1.0783, -0.6704]])

In [128]:
C[X] # this is the same as doing one-hot encoding and matrix multiplication for all the indices in X

tensor([[[ 1.0696, -0.5928],
         [ 1.0696, -0.5928],
         [ 1.0696, -0.5928]],

        [[ 1.0696, -0.5928],
         [ 1.0696, -0.5928],
         [ 0.1072, -0.7512]],

        [[ 1.0696, -0.5928],
         [ 0.1072, -0.7512],
         [-1.0783, -0.6704]],

        [[ 0.1072, -0.7512],
         [-1.0783, -0.6704],
         [-1.0783, -0.6704]],

        [[-1.0783, -0.6704],
         [-1.0783, -0.6704],
         [ 0.6969,  0.8277]],

        [[ 1.0696, -0.5928],
         [ 1.0696, -0.5928],
         [ 1.0696, -0.5928]],

        [[ 1.0696, -0.5928],
         [ 1.0696, -0.5928],
         [-0.0637, -0.8183]],

        [[ 1.0696, -0.5928],
         [-0.0637, -0.8183],
         [ 1.7781,  0.1825]],

        [[-0.0637, -0.8183],
         [ 1.7781,  0.1825],
         [-0.0149, -0.7220]],

        [[ 1.7781,  0.1825],
         [-0.0149, -0.7220],
         [-0.7590,  0.2823]],

        [[-0.0149, -0.7220],
         [-0.7590,  0.2823],
         [-0.0149, -0.7220]],

        [[-0.7590,  0

In [129]:
C[X].shape # (number of training examples, block_size, embedding dimension)

torch.Size([32, 3, 2])

In [130]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

# 2. Gizli katmanı ve çıkış katmanını kur: embedding'leri düzleştir, W1 ve b1 ile tanh, W2 ve b2 ile logits. Loss'u geçen haftaki gibi elle hesapla, sonra F.cross_entropy ile aynı sonucu aldığını göster ve neden onu tercih ettiğimizi videodan anla.

## 2.1 Construct the hidden layer

In [131]:
W1 = torch.randn((6, 100)) # 6 input features (3 characters * 2 embedding dimensions) and 100 output features (hidden layer size)
B1 = torch.randn(100) # bias for the hidden layer

In [132]:
# Basicly what we want to do is this
# W1 @ emb + B1
# But the emb is 3D and the W1 is 2D, so we need to reshape the emb to be 2D as well. We can do this by using the view method to flatten the last two dimensions of emb into a single dimension. This will give us a 2D tensor with shape (number of training examples, block_size * embedding dimension). Then we can do the matrix multiplication with W1 and add B1.

# Why Do We Concatenate Embeddings Before the Hidden Layer?

Suppose our input has:

* `batch_size = 32`
* `block_size = 3` characters
* `embedding_dim = 2`

After the embedding lookup, our tensor has the shape:

```python
(32, 3, 2)
```

This means:

```text
32 training examples
3 characters per example
2-dimensional embedding for each character
```

For example, one training example might look like:

```text
[
    [0.2, 0.5],    # character 1
    [0.1, -0.3],   # character 2
    [0.7, 0.8]     # character 3
]
```

## Why do we reshape it?

Our first hidden layer is defined as:

```python
W1 = torch.randn((6, 100))
B1 = torch.randn(100)
```

The `6` comes from:

```text
block_size × embedding_dim
= 3 × 2
= 6
```

So we transform:

```text
(batch_size, block_size, embedding_dim)
```

into:

```text
(batch_size, block_size * embedding_dim)
```

In our example:

```text
(32, 3, 2)
    ↓
(32, 6)
```

The embeddings are not reduced or lost. We are simply putting the embeddings of the three characters next to each other:

```text
[0.2, 0.5] + [0.1, -0.3] + [0.7, 0.8]

        ↓

[0.2, 0.5, 0.1, -0.3, 0.7, 0.8]
```

Now each training example is represented by a single 6-dimensional feature vector.

## Why do we need this?

Because we want the hidden layer to consider **all three characters together**.

The matrix multiplication is:

```text
X       @       W1       +       B1
(32,6)      (6,100)            (100,)

                ↓

             (32,100)
```

Therefore:

```python
hidden = X @ W1 + B1
```

The hidden layer receives information from all three characters simultaneously.

For example:

```text
character 1 ─┐
character 2 ─┼──→ concatenate → Linear(6, 100) → hidden layer
character 3 ─┘
```

## Important: We are NOT reducing the embedding dimension

* We are not reducing the embeddings; we are simply putting the 3 embeddings into one feature vector so the hidden layer can process the entire context together.

## In short

```text
Embedding output:

(batch_size, block_size, embedding_dim)
              ↓
            reshape
              ↓
(batch_size, block_size × embedding_dim)
              ↓
        Linear layer
              ↓
(batch_size, hidden_size)
```

For our example:

```text
(32, 3, 2)
    ↓
(32, 6)
    ↓
Linear(6, 100)
    ↓
(32, 100)
```

The main reason is:

> **We concatenate the embeddings so that the hidden layer can process the entire context window as one feature vector and learn interactions between all characters.**


In [133]:
h = torch.tanh(emb.view(-1, 6) @ W1 + B1) # flatten the last two dimensions of emb into a single dimension and do the matrix multiplication with W1 and add B1

In [134]:
h # (number of training examples, hidden layer size)

tensor([[-0.0065, -0.8648,  0.9840,  ..., -0.9549, -0.5578,  0.9999],
        [-0.0023, -0.9825,  0.9100,  ..., -0.9936, -0.4416,  0.9954],
        [ 0.6820, -0.9951,  0.0389,  ..., -0.9978, -0.5883, -0.4114],
        ...,
        [-0.9752,  0.8264, -0.9830,  ..., -0.8930,  0.9988,  0.8753],
        [-0.9997,  0.9950, -0.9910,  ..., -0.9734,  0.9956,  0.9688],
        [ 0.8390,  0.8756,  0.5958,  ..., -0.7809,  0.3514,  0.9963]])

In [135]:
h.shape # (number of training examples, hidden layer size)

torch.Size([32, 100])

## 2.2 Create the output layer

In [136]:
W2 = torch.randn((100, 27)) # 100 input features (hidden layer size) and 27 output features (number of characters in the vocabulary)
B2 = torch.randn(27) # bias for the output layer

In [137]:
logits = h @ W2 + B2 # (number of training examples, number of characters in the vocabulary)

In [138]:
logits.shape # (number of training examples, number of characters in the vocabulary)

torch.Size([32, 27])

In [139]:
counts = logits.exp() # apply the exponential function to the logits to get unnormalized probabilities
probs = counts / counts.sum(1, keepdims=True) # normalize the probabilities by dividing by the sum of each row

In [140]:
probs.shape # (number of training examples, number of characters in the vocabulary)

torch.Size([32, 27])

## 2.3 Create the loss function.

In [141]:
loss = -probs[torch.arange(32), Y].log().mean() # negative log likelihood loss, we take the log of the probabilities of the correct characters and take the mean over all training examples
loss

tensor(12.3467)

## Back from Scratch

In [142]:
X.shape, Y.shape # dataset

(torch.Size([32, 3]), torch.Size([32]))

In [143]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [144]:
sum(p.nelement() for p in parameters) # number of parameters in total

3481

In [145]:
# Forward pass
emb = C[X] # (32, 3, 2)
h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)
#counts = logits.exp()
#prob = counts / counts.sum(1, keepdims=True)
#loss = -prob[torch.arange(32), Y].log().mean()
loss = F.cross_entropy(logits, Y) # this is the same as the loss we calculated above, but it is more numerically stable and efficient
loss

tensor(17.7697)

## Why Use `F.cross_entropy`?

Instead of manually calculating:

```python
counts = logits.exp()
prob = counts / counts.sum(1, keepdims=True)
loss = -prob[torch.arange(32), Y].log().mean()
```

we use:

```python
loss = F.cross_entropy(logits, Y)
```

### Why?

1. **More efficient:** It combines the operations internally, using less memory and computation.

2. **Better backward pass:** PyTorch can optimize the gradient computation by combining the operations, making backpropagation more efficient.

3. **Numerically more stable:** It avoids explicitly computing `exp()` and probabilities, reducing the risk of numerical overflow/underflow.


# 3. Eğitim döngüsünü kur: önce tek bir minibatch'i overfit et, sonra bütün veriyi minibatch'lerle eğit. Learning rate'i videodaki gibi tara ve iyi bir değer seç. Veriyi train / dev / test olarak böl, loss'u dev üzerinde raporla.

## 3.1 Construct the full dataset

In [146]:
# build the full dataset
block_size = 3
X, Y = [], []
for w in words:
  #print(w)
  context = [0] * block_size
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    #print(''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix]
X = torch.tensor(X)
Y = torch.tensor(Y)

In [147]:
X.shape, Y.shape # dataset

(torch.Size([228146, 3]), torch.Size([228146]))

In [148]:
for p in parameters:
  p.requires_grad = True

In [149]:
for _ in range(10):
  # forward pass
  emb = C[X] # (32, 3, 2)
  h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 100)
  logits = h @ W2 + b2 # (32, 27)
  loss = F.cross_entropy(logits, Y)
  print(loss.item())
  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward()
  # update
  for p in parameters:
    p.data += -0.1 * p.grad


19.505229949951172
17.084495544433594
15.776529312133789
14.833335876464844
14.002598762512207
13.253257751464844
12.579916000366211
11.983101844787598
11.470492362976074
11.05185604095459
